## **1 кейс**

**Система автопроверки**

**Важно**

Перед началом решения выполните следующую ячейку, чтобы загрузить два файла - эталонный код и код пользователя.

In [ ]:
!wget https://gist.github.com/Vs8th/aafed81d81678c807a3ad3dbf93588b2/raw/user.py

!wget https://gist.github.com/Vs8th/95f7897019a4213c76e5b65234d31e30/raw/etalon.py

!wget https://gist.github.com/Vs8th/59f797dfbd33f9a4be1e5ed43bb42f4d/raw/res_cor.txt

--2026-04-29 06:24:31--  https://gist.github.com/Vs8th/aafed81d81678c807a3ad3dbf93588b2/raw/user.py
Resolving gist.github.com (gist.github.com)... 140.82.114.3
Connecting to gist.github.com (gist.github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/aafed81d81678c807a3ad3dbf93588b2/raw/user.py [following]
--2026-04-29 06:24:31--  https://gist.githubusercontent.com/Vs8th/aafed81d81678c807a3ad3dbf93588b2/raw/user.py
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 106 [text/plain]
Saving to: ‘user.py’

user.py             100%[===================>]     106  --.-KB/s    in 0s      

2026-04-29 06:24:31 (2.21 MB/s) - ‘user.py’ saved [106/106]

--2026-04-29 0

Давайте сразу посмотрим с каким кодом мы имеем дело.

In [ ]:
with open('etalon.py', 'r') as f:
    lines = f.readlines()

lines

['def add(a, b):\n', '    return a + b']

In [ ]:
with open('user.py', 'r') as f:
    lines2 = f.readlines()

lines2

['def add_numbers(a, b):\n',
 '    return a - b  # Ошибка: нужно сложить, а не вычесть']

### **Решения**

#### **Задача 1**

Задача не из легких, поэтому начнем по шагам. Для начала напишите функцию `run_tests`, которая прочтет эталонный код и код пользователя из `.py` файлов, выполнит их и сохранит результаты в файл `output.txt`.

**Решение**

Напишите свое решение ниже

In [ ]:

import importlib
import inspect


def get_function(module):
    for name, obj in inspect.getmembers(module):
        if inspect.isfunction(obj) and obj.__module__ == module.__name__:
            return obj


def run_tests(standard_code, user_code, test_cases, output_file):
    standard_func = get_function(standard_code)
    user_func = get_function(user_code)

    result = []

    for test_case in test_cases:
        try:
            expected = standard_func(*test_case)
            user_result = user_func(*test_case)

            if expected == user_result:
                result.append("Тест пройден")
            else:
                result.append("Тест не пройден")
                result.append(f"Ожидаемый результат: {expected}")
                result.append(f"Результат пользователя: {user_result}")

        except Exception as e:
            with open(output_file, 'w') as f:
                f.write(f"Ошибка при выполнении кода пользователя: {e}")
            return

    with open(output_file, 'w') as f:
        f.write("\n".join(result) + "\n")


# Чтение кодов из файлов
standard_module = importlib.import_module('etalon')
user_module = importlib.import_module('user')

test_cases = [(1, 2), (3, 4), (5, 6)]
output_file = 'output.txt'

run_tests(standard_module, user_module, test_cases, output_file)


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [ ]:
with open('output.txt', 'r') as file1:
    res1 = file1.read()

with open('res_cor.txt', 'r') as file2:
    res2 = file2.read()

try:
    assert res1 == res2
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 2**

Теперь давайте двигаться дальше. Нужно добавить проверку на вредоносный код. Напишите отдельную функцию - `check_for_malicious_code`, которую мы будем вызывать перед проверкой кода.

Перед написанием кода, выполните следующую ячейку, в ней скачивается новый код пользователя, который проверит работу функции `check_for_malicious_code`.

In [ ]:
!wget https://gist.github.com/Vs8th/9f6a2c73c755cfcc088c7835ec4fb2c0/raw/user_malicious.py

**Решение**

Напишите свое решение ниже

In [ ]:

import importlib
import inspect
import ast


FORBIDDEN_NAMES = ['eval', 'exec']


def check_for_malicious_code(code, output_file):
    try:
        tree = ast.parse(code)

        for node in ast.walk(tree):
            if isinstance(node, ast.Call):
                if isinstance(node.func, ast.Name):
                    if node.func.id == 'eval':
                        return "Код содержит вызов eval()"

            if isinstance(node, ast.Attribute):
                if node.attr in FORBIDDEN_NAMES:
                    return f"Код содержит запрещенный атрибут: {node.attr}"

    except SyntaxError as e:
        return f"Код содержит синтаксическую ошибку: {type(e).__name__}"

    except Exception as e:
        return f"Код содержит вредоносные элементы: {e}"

    return None


def get_function(module):
    for name, obj in inspect.getmembers(module):
        if inspect.isfunction(obj) and obj.__module__ == module.__name__:
            return obj


def run_tests(standard_code, user_code, test_cases, output_file):
    standard_func = get_function(standard_code)
    user_func = get_function(user_code)

    result = []

    for test_case in test_cases:
        try:
            expected = standard_func(*test_case)
            user_result = user_func(*test_case)

            if expected == user_result:
                result.append("Тест пройден")
            else:
                result.append("Тест не пройден")
                result.append(f"Ожидаемый результат: {expected}")
                result.append(f"Результат пользователя: {user_result}")

        except Exception as e:
            with open(output_file, 'w') as f:
                f.write(f"Ошибка при выполнении кода пользователя: {e}")
            return

    with open(output_file, 'w') as f:
        f.write("\n".join(result) + "\n")


# Ensure user_malicious.py is downloaded
!wget -q https://gist.github.com/Vs8th/9f6a2c73c755cfcc088c7835ec4fb2c0/raw/user_malicious.py -O user_malicious.py

# Чтение кодов из файлов
standard_module = importlib.import_module('etalon')
user_module = importlib.import_module('user_malicious')

test_cases = [(1, 2), (3, 4), (5, 6)]
output_file = 'output2.txt'


# Проверка на вредоносный код
malicious_code_message = check_for_malicious_code(inspect.getsource(user_module), output_file)

if malicious_code_message:
    with open(output_file, 'w') as f:
        f.write("Ошибка при проверке модуля user_malicious:\n")
        f.write(malicious_code_message)
else:
    run_tests(standard_module, user_module, test_cases, output_file)


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [ ]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/5197125c5ef1e1b34b7d73adccdaf4bb/raw/cor_output_2.txt

--2026-04-29 06:39:23--  https://gist.github.com/Vs8th/5197125c5ef1e1b34b7d73adccdaf4bb/raw/cor_output_2.txt
Resolving gist.github.com (gist.github.com)... 140.82.112.4
Connecting to gist.github.com (gist.github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/5197125c5ef1e1b34b7d73adccdaf4bb/raw/cor_output_2.txt [following]
--2026-04-29 06:39:24--  https://gist.githubusercontent.com/Vs8th/5197125c5ef1e1b34b7d73adccdaf4bb/raw/cor_output_2.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 107 [text/plain]
Saving to: ‘cor_output_2.txt.2’

cor_output_2.txt.2  100%[===================>]     107  --.-KB/s    in 0s      

2026-04-29 06:39:24 (4.16 MB/s) - ‘co

In [ ]:
with open('output2.txt', 'r') as file1:
    res1 = file1.read()

with open('cor_output_2.txt', 'r') as file2:
    res2 = file2.read()

try:
    assert res1== res2
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 3**

Движемся дальше - последнее, что нам нужно проверить - время выполнения. Если время выполнения превышает 5 секунд, уведомите об этом пользователя и остановите выполнение проверки.

Перед написанием кода, выполните следующую ячейку, в ней скачивается новый код пользователя, который проверит работу проверки на время выполнения.

In [ ]:
!wget https://gist.github.com/Vs8th/4aba49489a8b0843ef85c4a1a93f923d/raw/user_inf.py

**Решение**

Напишите свое решение ниже

In [37]:
import importlib
import inspect
import ast
import multiprocessing


FORBIDDEN_NAMES = ['eval']


def check_for_malicious_code(code, output_file):
    try:
        tree = ast.parse(code)

        for node in ast.walk(tree):
            if isinstance(node, ast.Call):
                if isinstance(node.func, ast.Name) and node.func.id == 'eval':
                    return "Код содержит вызов eval()"

    except SyntaxError as e:
        return f"Код содержит синтаксическую ошибку: {type(e).__name__}"

    except Exception as e:
        return f"Код содержит вредоносные элементы: {e}"

    return None


def get_function(module):
    for name, obj in inspect.getmembers(module):
        if inspect.isfunction(obj) and obj.__module__ == module.__name__:
            return obj


def run_user_function(user_func, test_case):
    user_func(*test_case)


def run_tests(standard_code, user_code, test_cases, output_file):
    user_func = get_function(user_code)

    process = multiprocessing.Process(
        target=run_user_function,
        args=(user_func, test_cases[0])
    )

    process.start()
    process.join(timeout=5)

    if process.is_alive():
        process.terminate()
        process.join()

        with open(output_file, 'w') as f:
            f.write("Предупреждение: время выполнения превышает 5 секунд.")
        return


# Чтение кодов из файлов
standard_module = importlib.import_module('etalon')
user_module = importlib.import_module('user_inf')

test_cases = [(1, 2), (3, 4), (5, 6)]
output_file = 'output3.txt'

# Проверка на вредоносный код
malicious_code_message = check_for_malicious_code(inspect.getsource(user_module), output_file)

if malicious_code_message:
    with open(output_file, 'w') as f:
        f.write("Ошибка при проверке модуля user_malicious:\n")
        f.write(malicious_code_message)
else:
    run_tests(standard_module, user_module, test_cases, output_file)


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [38]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/cfcafbd2cd46e8905555b02eae78406e/raw/cor_res_3.txt

--2026-04-29 06:51:28--  https://gist.github.com/Vs8th/cfcafbd2cd46e8905555b02eae78406e/raw/cor_res_3.txt
Resolving gist.github.com (gist.github.com)... 140.82.113.4
Connecting to gist.github.com (gist.github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/cfcafbd2cd46e8905555b02eae78406e/raw/cor_res_3.txt [following]
--2026-04-29 06:51:28--  https://gist.githubusercontent.com/Vs8th/cfcafbd2cd46e8905555b02eae78406e/raw/cor_res_3.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 96 [text/plain]
Saving to: ‘cor_res_3.txt.4’

cor_res_3.txt.4     100%[===================>]      96  --.-KB/s    in 0s      

2026-04-29 06:51:28 (2.04 MB/s) - ‘cor_res_3.txt.4

In [39]:
with open('output3.txt', 'r') as file1:
    res1 = file1.read()

with open('cor_res_3.txt', 'r') as file2:
    res2 = file2.read()

try:
    assert res1 == res2
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!
